## Prepare dataset and model

In [1]:
import torch

In [3]:
# CUDA support 
if torch.cuda.is_available():
    device = torch.device('cuda')
else: 
    device = torch.device('cpu')

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torchvision.models import resnet18
from tqdm import tqdm
import random
import torchvision
from sklearn.decomposition import PCA

In [5]:
from CUP import Cup

In [6]:

seed = 1234
torch.manual_seed(seed)
torch.cuda.manual_seed(seed) 
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [7]:
path = 'data path'

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
trainset = torchvision.datasets.CIFAR10(root=path, train=True, download=False, transform=transform)
testset = torchvision.datasets.CIFAR10(root=path, train=False, download=False, transform=transform)

In [8]:
def split_cifar10_by_class(dataset, forget_class):
    retain_indices = []
    forget_indices = []
    
    for i in range(len(dataset)):
        _, label = dataset[i]
        if label == forget_class:
            forget_indices.append(i)
        else:
            retain_indices.append(i)
    
    retain_set = torch.utils.data.Subset(dataset, retain_indices)
    forget_set = torch.utils.data.Subset(dataset, forget_indices)
    return retain_set, forget_set

forget_class = 0
retain_set, forget_set = split_cifar10_by_class(trainset, forget_class)


trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
forgetloader = torch.utils.data.DataLoader(forget_set, batch_size=128, shuffle=True, num_workers=2)
retainloader = torch.utils.data.DataLoader(retain_set, batch_size=128, shuffle=True, num_workers=2)


In [ ]:
model = resnet18(pretrained=False) 
model.fc = nn.Linear(model.fc.in_features, 10)  
model = model.to(device)


## Train then Unlearn

In [10]:
def train_model_collect_trajectory(model, trainloader, num_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    trajectory = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()


        current_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
        trajectory.append(current_params)
        if epoch%10 == 0:
            print(loss.item())

    final_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
    return np.array(trajectory), final_params


In [11]:
def unlearn_model_collect_trajectory(model, trainloader,lr=0.00001, num_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    trajectory = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = -criterion(outputs, labels)
            loss.backward()
            optimizer.step()


        current_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
        trajectory.append(current_params)
        print(-loss)

    final_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
    return np.array(trajectory), final_params


def ws_model_collect_trajectory(model, forgetloader, retainloader,  num_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.0005, momentum=0.9)
    trajectory = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        for (inputs_forget, labels_forget), (inputs_retain, labels_retain) in zip(forgetloader, retainloader):
            optimizer.zero_grad()
            inputs_forget, labels_forget = inputs_forget.to(device), labels_forget.to(device)
            outputs_forget = model(inputs_forget)
            forget_loss = -criterion(outputs_forget, labels_forget)
            inputs_retain, labels_retain = inputs_retain.to(device), labels_retain.to(device)
            outputs_retain = model(inputs_retain)
            retain_loss = criterion(outputs_retain, labels_retain)            
            total_loss = forget_loss + retain_loss
            total_loss.backward()
            optimizer.step()
        print(f'forgetloss:{-forget_loss}, retain_loss: {retain_loss}')
        current_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
        trajectory.append(current_params)

    final_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
    return np.array(trajectory), final_params


def cup_model_collect_trajectory(model, forgetloader, retainloader,  num_epochs=5):
    criterion = nn.CrossEntropyLoss()
    base_optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.9)
    cup_optimizer = Cup(base_optimizer, alpha=0.01)
    trajectory = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        for (inputs_forget, labels_forget), (inputs_retain, labels_retain) in zip(forgetloader, retainloader):
            cup_optimizer.zero_grad()
            inputs_forget, labels_forget = inputs_forget.to(device), labels_forget.to(device)
            outputs_forget = model(inputs_forget)
            forget_loss = -criterion(outputs_forget, labels_forget)
            inputs_retain, labels_retain = inputs_retain.to(device), labels_retain.to(device)
            outputs_retain = model(inputs_retain)
            retain_loss = criterion(outputs_retain, labels_retain)            
            cup_optimizer.step([forget_loss, retain_loss])
        print(f'forgetloss:{-forget_loss}, retain_loss: {retain_loss}')
        current_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
        trajectory.append(current_params)

    final_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()
    return np.array(trajectory), final_params

In [ ]:
initial_trajectory, initial_final_params = train_model_collect_trajectory(model, trainloader, num_epochs=200)

In [13]:
import copy

In [14]:
unlearn_model = copy.deepcopy(model)

In [ ]:
re_trajectory, re_final_params = unlearn_model_collect_trajectory(unlearn_model,retainloader,lr=0.00001, num_epochs=100)

In [16]:
unlearn_model = copy.deepcopy(model)

In [ ]:
ga_trajectory, ga_final_params = unlearn_model_collect_trajectory(unlearn_model, forgetloader,lr=0.0005, num_epochs=10)

In [18]:
ws_model = copy.deepcopy(model)

In [ ]:
ws_trajectory, ws_final_params = ws_model_collect_trajectory(ws_model, forgetloader,retainloader, num_epochs=10)

In [20]:
cup_model = copy.deepcopy(model)

In [ ]:
cup_trajectory, cup_final_params = cup_model_collect_trajectory(cup_model, forgetloader,retainloader, num_epochs=10)

In [22]:
cup_trajectory = np.insert(cup_trajectory,0,initial_final_params, axis=0)
ws_trajectory = np.insert(ws_trajectory,0,initial_final_params, axis=0)
ga_trajectory = np.insert(ga_trajectory,0,initial_final_params, axis=0)

## Visualize optimization trajectory

In [24]:
def perform_pca_with_centering(trajectory, center):
    centered_trajectory = np.array([param - center for param in trajectory])
    pca = PCA(n_components=2)
    pca.fit(centered_trajectory)
    pc1 = pca.components_[0]
    pc2 = pca.components_[1]
    return pc1, pc2

In [25]:
g1, g2 = perform_pca_with_centering(re_trajectory, re_trajectory[-1])

In [26]:
w0 = re_trajectory[-1]

In [28]:
def evaluate_loss_on_gradients(g1,g2,w0, dataloader, grid_size=12, range_lim=12, if_forget=False):
    W1, W2 = np.meshgrid(np.linspace(-range_lim, 5, grid_size), np.linspace(-range_lim, range_lim, grid_size))
    loss_grid = np.zeros_like(W1)
    w0 = torch.tensor(w0, dtype=torch.float32).to(device)
    for i in tqdm(range(grid_size)):
        for j in range(grid_size):

            perturbation = W1[i, j] * g1 + W2[i, j] * g2
            perturbed_params = w0 + torch.tensor(perturbation, dtype=torch.float32).to(device)

            torch.nn.utils.vector_to_parameters(perturbed_params, model.parameters())

            model.eval()
            total_loss = 0.0
            criterion = nn.CrossEntropyLoss()
            with torch.no_grad():
                for inputs, labels in dataloader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_loss += loss.item() * inputs.size(0)
            if if_forget:
                loss_grid[i, j] = -total_loss / len(dataloader.dataset)
            else:
                loss_grid[i, j] = total_loss / len(dataloader.dataset)

  
    torch.nn.utils.vector_to_parameters(w0, model.parameters())

    return W1, W2, loss_grid

In [29]:
import random
from torch.utils.data import Subset
from torch.utils.data import DataLoader

In [30]:
def create_sampled_dataloader(dataloader, sample_size):

    sampled_indices = random.sample(range(len(dataloader.dataset)), sample_size)
    sampled_dataset = Subset(dataloader.dataset, sampled_indices)
    sampled_loader = DataLoader(sampled_dataset, batch_size=sample_size, shuffle=False)
    return sampled_loader

In [31]:
forget_size = len(forgetloader.dataset)//2

trainloader_sampled = create_sampled_dataloader(trainloader, forget_size)
retainloader_sampled = create_sampled_dataloader(retainloader, forget_size)
forgetloader_sampled = create_sampled_dataloader(forgetloader, forget_size)

In [ ]:
W1_total, W2_total, total_loss_grid = evaluate_loss_on_gradients(g1, g2,initial_final_params, trainloader_sampled)
W1_retain, W2_retain, retain_loss_grid = evaluate_loss_on_gradients(g1, g2,initial_final_params, retainloader_sampled)
W1_forget, W2_forget, forget_loss_grid = evaluate_loss_on_gradients(g1, g2,initial_final_params, forgetloader_sampled)


In [33]:
def project_trajectory_onto_pcs(trajectory, pc1, pc2):
    projected_pc1 = np.dot(trajectory - initial_final_params, pc1)
    projected_pc2 = np.dot(trajectory - initial_final_params, pc2)
    return projected_pc1, projected_pc2

In [34]:
def project_trajectory_onto_plane(trajectory, v1, v2, origin=None):

    if origin is None:
        origin = trajectory[0]

    trajectory = trajectory - origin
    projected_v1 = np.dot(trajectory, v1)
    projected_v2 = np.dot(trajectory, v2)

    return projected_v1, projected_v2

In [35]:
cup_proj1, cup_proj2 = project_trajectory_onto_pcs(cup_trajectory, g1, g2)
ga_proj1, ga_proj2 = project_trajectory_onto_pcs(ga_trajectory, g1, g2)
ws_proj1, ws_proj2 = project_trajectory_onto_pcs(ws_trajectory, g1, g2)

In [ ]:
xlim = (-12, 5)
ylim = (-12, 12)

title_fontsize = 20
label_fontsize = 20
color_level = 40

original_params = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy()


fig, ax = plt.subplots(figsize=(6, 5))

vmin_total = total_loss_grid.min()
vmax_total = total_loss_grid.max()

contour1 = ax.contourf(W1_total, W2_total, total_loss_grid, levels=color_level, cmap='RdYlBu', vmin=vmin_total, vmax=vmax_total)

ax.set_xlabel(r'$v_1$', fontsize=label_fontsize)
ax.set_ylabel(r'$v_2$', fontsize=label_fontsize)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.tick_params(axis='both', which='major', labelsize=20)


ax.plot(cup_proj1[0], cup_proj2[0], 'ko', markersize=7, label='Pre-trained Params')


ax.plot(ga_proj1[:], ga_proj2[:], 'o-', color='purple', markersize=3, label='GA Trajectory')
ax.plot(ga_proj1[-1], ga_proj2[-1], '*', color='purple', markersize=10, label='Final GA Params')

ax.plot(ws_proj1[:6], ws_proj2[:6], 'o-', color='green', markersize=3, label='WS Trajectory')
ax.plot(ws_proj1[5], ws_proj2[5], '*', color='green', markersize=10, label='Final WS Params')

ax.plot(cup_proj1[:], cup_proj2[:], 'o-', color='blue', markersize=3, label='CUP Trajectory')
ax.plot(cup_proj1[-1], cup_proj2[-1], 'b*', markersize=10, label='Final CUP Params')

ax.legend(fontsize=10, loc='upper right')
cbar = plt.colorbar(contour1, ax=ax)
cbar.set_label(r'$\mathcal{L}_T(\theta_t)$',fontsize=label_fontsize)

plt.savefig('figure/total_set_loss.pdf', bbox_inches='tight')
plt.savefig('figure/total_set_loss.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))


vmin_retain = retain_loss_grid.min()
vmax_retain = retain_loss_grid.max()

contour2 = ax.contourf(W1_retain, W2_retain, retain_loss_grid, levels=color_level, cmap='RdYlBu', vmin=vmin_total, vmax=vmax_total)

ax.set_xlabel(r'$v_1$', fontsize=label_fontsize)
ax.set_ylabel(r'$v_2$', fontsize=label_fontsize)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.tick_params(axis='both', which='major', labelsize=20)


ax.plot(cup_proj1[0], cup_proj2[0], 'ko', markersize=7, label='Pre-trained Params')

ax.plot(ga_proj1[:], ga_proj2[:], 'o-', color='purple', markersize=3, label='GA Trajectory')
ax.plot(ga_proj1[-1], ga_proj2[-1], '*', color='purple', markersize=10, label='Final GA Params')

ax.plot(ws_proj1[:6], ws_proj2[:6], 'o-', color='green', markersize=3, label='WS Trajectory')
ax.plot(ws_proj1[5], ws_proj2[5], '*', color='green', markersize=10, label='Final WS Params')

ax.plot(cup_proj1[:], cup_proj2[:], 'o-', color='blue', markersize=3, label='CUP Trajectory')
ax.plot(cup_proj1[-1], cup_proj2[-1], 'b*', markersize=10, label='Final CUP Params')


ax.legend(fontsize=10, loc='upper right')
cbar = plt.colorbar(contour2, ax=ax)
cbar.set_label(r'$\mathcal{L}_r(\theta_t)$', fontsize=label_fontsize)


plt.savefig('figure/retain_set_loss.pdf', bbox_inches='tight')
plt.savefig('figure/retain_set_loss.png', bbox_inches='tight')
plt.show()


fig, ax = plt.subplots(figsize=(6, 5))

vmax_forget = -forget_loss_grid.min()
vmin_forget = -forget_loss_grid.max()

contour3 = ax.contourf(W1_forget, W2_forget, -forget_loss_grid, levels=color_level, cmap='RdYlBu', vmin=vmin_forget, vmax=vmax_forget)

ax.set_xlabel(r'$v_1$', fontsize=label_fontsize)
ax.set_ylabel(r'$v_2$', fontsize=label_fontsize)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.tick_params(axis='both', which='major', labelsize=20)

# Add trajectories
ax.plot(cup_proj1[0], cup_proj2[0], 'ko', markersize=7, label='Pre-trained Params')

ax.plot(ga_proj1[:], ga_proj2[:], 'o-', color='purple', markersize=3, label='GA Trajectory')
ax.plot(ga_proj1[-1], ga_proj2[-1], '*', color='purple', markersize=10, label='Final GA Params')

ax.plot(ws_proj1[:6], ws_proj2[:6], 'o-', color='green', markersize=3, label='WS Trajectory')
ax.plot(ws_proj1[5], ws_proj2[5], '*', color='green', markersize=10, label='Final WS Params')

ax.plot(cup_proj1[:], cup_proj2[:], 'o-', color='blue', markersize=3, label='CUP Trajectory')
ax.plot(cup_proj1[-1], cup_proj2[-1], 'b*', markersize=10, label='Final CUP Params')

# Add legend and colorbar
ax.legend(fontsize=10, loc='upper right')
cbar = plt.colorbar(contour3, ax=ax)
cbar.set_label(r'$\mathcal{L}_f(\theta_t)$',fontsize=label_fontsize)

# Save the figure
plt.savefig('figure/forget_set_loss.pdf',bbox_inches='tight')
plt.savefig('figure/forget_set_loss.png',bbox_inches='tight')
plt.show()